# PySpark Coding Challenges for Interviews

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/06_pyspark_coding_challenges.ipynb)

Five open-ended problems in the style of data-engineering interviews: deduplication, grain-aware joins,
as-of joins, sessionization and skewed joins. Each problem has fixtures, a statement with requirements, and a
reference solution with `assert`s.

**How to practise.** Read the statement and state the grain of every input and output out loud. Write your own
solution in a new cell under the fixtures and run it. Only then run the reference solution and compare. Do not
call `collect()` on the real data in an interview: the reference cells collect only tiny results to check them.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import time

# Workers must use this notebook's interpreter, and every timestamp here is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()

from pyspark.sql import SparkSession, Window, functions as F

spark = (
    SparkSession.builder.appName("pyspark-challenges")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # tiny data needs few partitions
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
MAX_TS = F.lit("9999-12-31 00:00:00").cast("timestamp")
print("Spark", spark.version)

## Problem 1 - Latest customer record with deterministic deduplication

The customer feed contains retries and out-of-order updates. Return exactly one row per `customer_id`, choosing
the latest `updated_at`. If timestamps tie, choose the greatest `ingest_id`. Keep customers whose email is null.

Requirements:

- Do not use `dropDuplicates` without defining which record wins.
- Say how the result changes if two records share `customer_id`, `updated_at` and `ingest_id`. Then the winner is
  arbitrary, so the pipeline should fail a check rather than pick one silently.
- Add a data-quality check that no `customer_id` has two winning records.

**Expected grain:** one row per customer.

In [ ]:
customer_updates = spark.createDataFrame([
    ('c1', 'ava@old.example', '2026-10-01 09:00:00', 10),
    ('c1', 'ava@example.com', '2026-10-01 09:00:00', 11),
    ('c2', None, '2026-10-02 10:00:00', 20),
    ('c2', 'ben@example.com', '2026-10-01 10:00:00', 19),
    ('c3', 'cy@example.com', '2026-10-03 12:00:00', 30)
], ['customer_id', 'email', 'updated_at', 'ingest_id'])
customer_updates.show(truncate=False)

Reference solution. Window on the key, order by the tie-break chain, keep rank 1.

In [ ]:
latest_customers = (
    customer_updates
    .withColumn("updated_ts", F.to_timestamp("updated_at"))
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_id")
            .orderBy(F.col("updated_ts").desc(), F.col("ingest_id").desc())
        ),
    )
    .filter(F.col("rn") == 1)
    .drop("rn", "updated_ts")
)
latest_customers.orderBy("customer_id").show(truncate=False)

rows = {r["customer_id"]: r for r in latest_customers.collect()}
assert latest_customers.count() == 3
assert rows["c1"]["ingest_id"] == 11 and rows["c1"]["email"] == "ava@example.com"
assert rows["c2"]["email"] is None                 # null email is kept, not dropped
assert rows["c3"]["ingest_id"] == 30

# Data-quality gate: exactly one winning record per key.
duplicate_winners = latest_customers.groupBy("customer_id").count().filter(F.col("count") > 1)
assert duplicate_winners.count() == 0

## Problem 2 - Orders joined to a many-to-many category dimension

Produce one row per order with `order_total`, `discount_total` and `net_total`. Each order has line items, and a
product can carry several category labels. The labels feed a separate category summary, but they must not
multiply order revenue.

Requirements:

- Compute line revenue at the line grain before any join to categories.
- Keep orders whose product has no category.
- Build the category summary separately, one row per category. A line in two categories counts in both, so
  category totals are not additive. Say so in the answer.
- Prove that adding the category relationship leaves the sum of order totals unchanged.

**Expected grains:** one row per order for the order summary; one row per category for the category summary.

In [ ]:
order_lines = spark.createDataFrame([
    ('o1', 'p1', 2, 10.0), ('o1', 'p2', 1, 25.0),
    ('o2', 'p1', 1, 10.0), ('o3', 'p3', 1, 8.0)
], ['order_id', 'product_id', 'quantity', 'unit_price'])
discounts = spark.createDataFrame([
    ('o1', 'd1', 5.0), ('o1', 'd2', 2.0), ('o3', 'd3', 1.0)
], ['order_id', 'discount_id', 'discount_amount'])
product_categories = spark.createDataFrame([
    ('p1', 'home'), ('p1', 'sale'), ('p2', 'home')
], ['product_id', 'category'])
order_lines.show(); discounts.show(); product_categories.show()

Reference solution. Revenue is computed per line, discounts are pre-aggregated to one row per order, and only then
are the joins made. That keeps the join from fanning out the order rows.

In [ ]:
lines = order_lines.withColumn("line_revenue", F.col("quantity") * F.col("unit_price"))

discount_by_order = discounts.groupBy("order_id").agg(F.sum("discount_amount").alias("discount_total"))

order_summary = (
    lines.groupBy("order_id").agg(F.sum("line_revenue").alias("order_total"))
    .join(discount_by_order, "order_id", "left")
    .withColumn("discount_total", F.coalesce("discount_total", F.lit(0.0)))
    .withColumn("net_total", F.col("order_total") - F.col("discount_total"))
    .orderBy("order_id")
)
order_summary.show()

category_summary = (
    lines.join(product_categories, "product_id", "inner")
    .groupBy("category").agg(F.sum("line_revenue").alias("category_revenue"))
    .orderBy("category")
)
category_summary.show()

by_order = {r["order_id"]: r for r in order_summary.collect()}
assert order_summary.count() == 3                  # o3 is kept although its product has no category
assert by_order["o1"]["order_total"] == 45.0 and by_order["o1"]["net_total"] == 38.0
assert by_order["o2"]["discount_total"] == 0.0
assert by_order["o3"]["net_total"] == 7.0

by_category = {r["category"]: r["category_revenue"] for r in category_summary.collect()}
assert by_category == {"home": 55.0, "sale": 30.0}

# Conservation: the category join does not change order revenue.
order_revenue = order_summary.agg(F.sum("order_total")).first()[0]
line_revenue = lines.agg(F.sum("line_revenue")).first()[0]
assert order_revenue == line_revenue == 63.0

## Problem 3 - Slowly changing dimension as-of join

Attach the customer segment that was valid when each event occurred. Dimension versions use the half-open interval
`[effective_from, effective_to)`, and a null `effective_to` means the version is current.

Requirements:

- Keep events that have no matching version.
- An event exactly at `effective_to` belongs to the next version.
- Detect overlapping versions of the same customer before the join, because overlaps would fan out the result.
- Return exactly one row per event.

**Expected grain:** one row per event.

In [ ]:
events = spark.createDataFrame([
    ('e1', 'c1', '2026-01-15 12:00:00', 20.0),
    ('e2', 'c1', '2026-07-01 12:00:00', 30.0),
    ('e3', 'c2', '2026-03-01 12:00:00', 40.0),
    ('e4', 'missing', '2026-03-01 12:00:00', 50.0)
], ['event_id', 'customer_id', 'event_ts', 'amount'])
segment_history = spark.createDataFrame([
    ('c1', 'standard', '2026-01-01 00:00:00', '2026-06-01 00:00:00'),
    ('c1', 'premium', '2026-06-01 00:00:00', None),
    ('c2', 'standard', '2026-01-01 00:00:00', None)
], ['customer_id', 'segment', 'effective_from', 'effective_to'])
events.show(); segment_history.show()

Reference solution. Two versions overlap when, for the earlier one `a` and the later one `b` of the same customer,
`b.effective_from` falls before `a.effective_to`. Check that first. Then join on the half-open interval.

In [ ]:
history = (segment_history
    .withColumn("effective_from", F.to_timestamp("effective_from"))
    .withColumn("effective_to", F.to_timestamp("effective_to")))
events_ts = events.withColumn("event_ts", F.to_timestamp("event_ts"))

a = history.alias("a")
b = history.alias("b")
overlapping = a.join(
    b,
    (F.col("a.customer_id") == F.col("b.customer_id"))
    & (F.col("a.effective_from") < F.col("b.effective_from"))
    & (F.col("b.effective_from") < F.coalesce(F.col("a.effective_to"), MAX_TS)),
)
assert overlapping.count() == 0, "overlapping segment versions would fan out the join"

in_version = (
    (F.col("e.customer_id") == F.col("h.customer_id"))
    & (F.col("e.event_ts") >= F.col("h.effective_from"))
    & (F.col("e.event_ts") < F.coalesce(F.col("h.effective_to"), MAX_TS))
)
priced = (events_ts.alias("e")
    .join(history.alias("h"), in_version, "left")
    .select("e.event_id", "e.customer_id", "e.event_ts", "e.amount", "h.segment"))
priced.orderBy("event_id").show(truncate=False)

segments = {r["event_id"]: r["segment"] for r in priced.collect()}
assert priced.count() == 4                         # one row per event, no fan-out
assert segments == {"e1": "standard", "e2": "premium", "e3": "standard", "e4": None}

# Boundary: an event exactly at effective_to belongs to the next version.
boundary = (spark.createDataFrame([("eb", "c1", "2026-06-01 00:00:00", 1.0)],
                                  ["event_id", "customer_id", "event_ts", "amount"])
    .withColumn("event_ts", F.to_timestamp("event_ts")))
boundary_row = (boundary.alias("e")
    .join(history.alias("h"), in_version, "left")
    .select("h.segment").first())
assert boundary_row["segment"] == "premium"

## Problem 4 - Sessionize events without leaking across users

Create sessions from web events. A new session starts when the gap from the previous event of the same user is
greater than 30 minutes (exactly 30 minutes stays in the session). Return one row per session with its start,
end, event count and total value.

Requirements:

- Window by user and order by timestamp, then event id, so the result is deterministic.
- Events of one user must never affect another user's sessions.
- A null value counts as zero in the session total.
- Explain the cost of the window and how you would scale it when one user has billions of events.

**Expected grain:** one row per user session.

In [ ]:
web_events = spark.createDataFrame([
    ('u1', 'a', '2026-10-01 10:00:00', 4.0),
    ('u1', 'b', '2026-10-01 10:20:00', None),
    ('u1', 'c', '2026-10-01 11:00:01', 7.0),
    ('u2', 'd', '2026-10-01 10:05:00', 3.0),
    ('u2', 'e', '2026-10-01 10:35:00', 5.0)
], ['user_id', 'event_id', 'event_ts', 'value'])
web_events.show()

Reference solution. `lag` finds the previous event, a flag marks each session start, and a running sum of the flag
numbers the sessions. The window is partitioned by user, so no session can cross users. The running sum is a
window over the user's full history: that is the cost to explain, because one very large user is one partition.

In [ ]:
ordered = Window.partitionBy("user_id").orderBy("event_ts", "event_id")
events_ts = web_events.withColumn("event_ts", F.to_timestamp("event_ts"))

with_gap = events_ts.withColumn("prev_ts", F.lag("event_ts").over(ordered))
is_new_session = F.when(
    F.col("prev_ts").isNull()
    | (F.unix_timestamp("event_ts") - F.unix_timestamp("prev_ts") > 30 * 60),
    1,
).otherwise(0)

sessions = (
    with_gap.withColumn("is_new", is_new_session)
    .withColumn("session_no", F.sum("is_new").over(ordered.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .groupBy("user_id", "session_no")
    .agg(
        F.min("event_ts").alias("session_start"),
        F.max("event_ts").alias("session_end"),
        F.count("*").alias("event_count"),
        F.sum(F.coalesce("value", F.lit(0.0))).alias("session_value"),
    )
    .orderBy("user_id", "session_no")
)
sessions.show(truncate=False)

rows = [tuple(r) for r in sessions.select("user_id", "event_count", "session_value").collect()]
assert rows == [("u1", 2, 4.0), ("u1", 1, 7.0), ("u2", 2, 8.0)]

## Problem 5 - Skewed join with selective salting

A fact table is joined to a dimension on `account_id`. Profiling shows that `UNKNOWN` holds most fact rows, so one
task would do most of the join work. Implement a salted join that gives the same result as the unsalted join.

Requirements:

- Salt only the hot key, with a deterministic bucket from `event_id`.
- Replicate only the hot dimension row across the salt range, not the whole dimension.
- Compare row counts and totals with a baseline unsalted inner join.
- Say when broadcasting the dimension or enabling AQE skew join would be better.
- Say how to choose the salt count from Spark UI metrics (task durations and shuffle read sizes), not by guessing.

**Expected grain:** one row per matching fact row.

In [ ]:
fact_events = spark.createDataFrame([
    ('e1', 'UNKNOWN', 10), ('e2', 'UNKNOWN', 20), ('e3', 'UNKNOWN', 30),
    ('e4', 'a1', 5), ('e5', 'a2', 7), ('e6', None, 11)
], ['event_id', 'account_id', 'metric'])
accounts = spark.createDataFrame([
    ('UNKNOWN', 'fallback'), ('a1', 'alpha'), ('a2', 'beta')
], ['account_id', 'account_type'])
fact_events.show(); accounts.show()

Reference solution. Hot-key fact rows get a salt from `pmod(hash(event_id), N)`. The hot dimension row is
cross-joined with the salt range, and cold keys keep salt 0. The join then matches on `(account_id, salt)`.

In [ ]:
SALT_BUCKETS = 3
HOT_KEY = "UNKNOWN"

baseline = fact_events.join(accounts, "account_id", "inner")

fact_salted = fact_events.withColumn(
    "salt",
    F.when(F.col("account_id") == HOT_KEY, F.pmod(F.hash("event_id"), F.lit(SALT_BUCKETS)))
    .otherwise(F.lit(0)),
)
salts = spark.range(SALT_BUCKETS).select(F.col("id").cast("int").alias("salt"))
hot_dimension = accounts.filter(F.col("account_id") == HOT_KEY).crossJoin(salts)
cold_dimension = accounts.filter(F.col("account_id") != HOT_KEY).withColumn("salt", F.lit(0))
dimension_salted = hot_dimension.unionByName(cold_dimension)

salted = (
    fact_salted.join(dimension_salted, ["account_id", "salt"], "inner")
    .select("event_id", "account_id", "metric", "account_type")
)

# Baseline equality: same rows, same count, same total.
assert salted.count() == baseline.count() == 5     # the null-account row is dropped by both
assert salted.agg(F.sum("metric")).first()[0] == baseline.agg(F.sum("metric")).first()[0] == 72
assert {r.event_id for r in salted.collect()} == {r.event_id for r in baseline.collect()}

## Interviewer notes

Keep the reference solutions and adversarial fixtures outside the candidate copy. Change values, column names,
duplicate patterns and null placement, but keep the schema stable. Judge the answer on grain, duplicate resistance,
null semantics, conservation of totals, deterministic tie-breaking, the physical plan (shuffles, broadcast, skew)
and testing, not only on whether the sample output looks right. Ask the candidate to rerun after changing the salt
count, or after adding a duplicate dimension record.